# Tahap 4: Bangun Case Base CBR

Notebook ini memproses file teks menjadi struktur kasus terstruktur untuk **Case-Based Reasoning (CBR)**.

Setiap kasus memiliki komponen:
- `problem` — pokok perkara / duduk perkara
- `solution` — amar putusan
- `dasar_hukum` — pasal-pasal yang dikutip
- `fitur` — kata kunci, jenis perkara, pengadilan (untuk similarity matching)

- **Input:** `../data/processed/teks_ekstrak/*.txt` (hasil Tahap 3)
- **Output:**
  - `../data/processed/case_base/case_base_[timestamp].json`
  - `../data/processed/case_base/case_base_[timestamp].csv`

In [1]:
!pip install pandas tqdm -q
print('Instalasi selesai!')

Instalasi selesai!


'pip' is not recognized as an internal or external command,
operable program or batch file.


## Konfigurasi

In [2]:
import os
import re
import json
import glob
import pandas as pd
from collections import Counter
from tqdm.notebook import tqdm
from datetime import datetime

TXT_DIR       = '../data/processed/teks_ekstrak'
CASE_BASE_DIR = '../data/processed/case_base'
os.makedirs(CASE_BASE_DIR, exist_ok=True)

print('Konfigurasi dimuat.')
print('Input TXT      : ' + os.path.abspath(TXT_DIR))
print('Output case base: ' + os.path.abspath(CASE_BASE_DIR))

Konfigurasi dimuat.
Input TXT      : c:\cbrs\data\processed\teks_ekstrak
Output case base: c:\cbrs\data\processed\case_base


## Fungsi Ekstraksi Komponen Kasus

In [3]:
STOPWORDS = {
    'yang', 'dan', 'di', 'ke', 'dari', 'untuk', 'dengan', 'pada', 'adalah',
    'ini', 'itu', 'dalam', 'oleh', 'sebagai', 'atau', 'bahwa', 'telah',
    'tidak', 'akan', 'juga', 'serta', 'tersebut', 'dapat', 'harus', 'sudah',
    'atas', 'maka', 'kepada', 'hal', 'juga', 'para', 'pihak', 'nomor',
}


def baca_file_teks(path_file):
    with open(path_file, 'r', encoding='utf-8') as f:
        isi = f.read()
    baris = isi.split('\n')
    meta, teks_baris, after_sep = {}, [], False
    for b in baris:
        if '=' * 10 in b:
            after_sep = True
            continue
        if not after_sep and ':' in b:
            kunci, _, nilai = b.partition(':')
            meta[kunci.strip()] = nilai.strip()
        elif after_sep:
            teks_baris.append(b)
    return meta, '\n'.join(teks_baris)


def ekstrak_amar(teks):
    m = re.search(
        r'(?:MENGADILI|M E N G A D I L I|Amar Putusan)(.*?)(?:Demikian|PENUTUP|$)',
        teks, re.DOTALL | re.IGNORECASE
    )
    if m:
        return m.group(1).strip()[:500]
    for pola in [r'(Mengabulkan gugatan.*?;)', r'(Menolak gugatan.*?;)', r'(Menyatakan.*?terbukti.*?;)']:
        m2 = re.search(pola, teks, re.IGNORECASE | re.DOTALL)
        if m2:
            return m2.group(1).strip()[:300]
    return ''


def ekstrak_pokok_perkara(teks):
    for pola in [
        r'(?:DUDUK PERKARA|Duduk Perkara|TENTANG DUDUK)(.*?)(?:TENTANG HUKUM|PERTIMBANGAN|$)',
        r'(?:bahwa|Bahwa)\s+(penggugat|pemohon|tergugat).*?(?:;|\n\n)',
    ]:
        m = re.search(pola, teks, re.DOTALL | re.IGNORECASE)
        if m:
            return m.group(1).strip()[:500]
    return teks[:300]


def ekstrak_dasar_hukum(teks):
    pasal_list = re.findall(
        r'(?:Pasal|pasal)\s+\d+[\s\w]*(?:KUHPerdata|KUH Perdata|HIR|RBg|UU|Undang-Undang)[\w\s.-]*',
        teks
    )
    return ' | '.join(list(dict.fromkeys(pasal_list))[:10])


def ekstrak_kata_kunci(teks, top_n=10):
    kata = re.findall(r'\b[a-zA-Z]{4,}\b', teks.lower())
    bersih = [k for k in kata if k not in STOPWORDS]
    return [k for k, _ in Counter(bersih).most_common(top_n)]


print('Fungsi ekstraksi komponen kasus siap.')

Fungsi ekstraksi komponen kasus siap.


## Bangun Case Base

In [4]:
txt_files = sorted(glob.glob(os.path.join(TXT_DIR, '*.txt')))

if not txt_files:
    print('Tidak ada file TXT di ' + TXT_DIR)
    print('Jalankan notebook 03_ekstrak_teks.ipynb terlebih dahulu.')
else:
    print('=' * 60)
    print('  MEMBANGUN CASE BASE DARI ' + str(len(txt_files)) + ' FILE')
    print('=' * 60)

    case_base = []

    for path_file in tqdm(txt_files, desc='Proses kasus'):
        nama_file = os.path.basename(path_file)
        try:
            meta, teks = baca_file_teks(path_file)

            amar        = ekstrak_amar(teks)
            pokok       = ekstrak_pokok_perkara(teks)
            dasar_hukum = ekstrak_dasar_hukum(teks)
            kata_kunci  = ekstrak_kata_kunci(teks)

            kasus = {
                'case_id'     : nama_file.replace('.txt', ''),
                'nomor'       : meta.get('NOMOR PUTUSAN', ''),
                'tanggal'     : meta.get('TANGGAL', ''),
                'pengadilan'  : meta.get('PENGADILAN', ''),
                'para_pihak'  : meta.get('PARA PIHAK', ''),
                'kategori'    : meta.get('KATEGORI', ''),
                'klasifikasi' : meta.get('KLASIFIKASI', ''),
                'url'         : meta.get('URL', ''),
                'problem'     : pokok[:500],
                'solution'    : amar[:500],
                'dasar_hukum' : dasar_hukum[:300],
                'teks_ringkas': teks[:1500],
                'fitur': {
                    'jenis_perkara': meta.get('KATEGORI', ''),
                    'pengadilan'   : meta.get('PENGADILAN', ''),
                    'ada_amar'     : bool(amar),
                    'panjang_teks' : len(teks),
                    'kata_kunci'   : kata_kunci,
                }
            }
            case_base.append(kasus)
        except Exception as e:
            print('  Gagal: ' + nama_file + ' — ' + str(e))

    print()
    print('Case base dibangun: ' + str(len(case_base)) + ' kasus')

  MEMBANGUN CASE BASE DARI 55 FILE


Proses kasus:   0%|          | 0/55 [00:00<?, ?it/s]


Case base dibangun: 55 kasus


## Simpan Case Base

In [5]:
if not case_base:
    print('Case base kosong.')
else:
    ts = datetime.now().strftime('%Y%m%d_%H%M%S')

    # Simpan JSON (lengkap dengan fitur)
    path_json = os.path.join(CASE_BASE_DIR, 'case_base_' + ts + '.json')
    with open(path_json, 'w', encoding='utf-8') as f:
        json.dump(case_base, f, ensure_ascii=False, indent=2)

    # Simpan CSV (flat, tanpa kolom fitur dict)
    rows_flat = []
    for k in case_base:
        rows_flat.append({
            'case_id'    : k['case_id'],
            'nomor'      : k['nomor'],
            'tanggal'    : k['tanggal'],
            'pengadilan' : k['pengadilan'],
            'para_pihak' : k['para_pihak'],
            'kategori'   : k['kategori'],
            'problem'    : k['problem'],
            'solution'   : k['solution'],
            'dasar_hukum': k['dasar_hukum'],
            'kata_kunci' : ', '.join(k['fitur']['kata_kunci']),
            'url'        : k['url'],
        })
    df_cbr = pd.DataFrame(rows_flat)
    path_csv = os.path.join(CASE_BASE_DIR, 'case_base_' + ts + '.csv')
    df_cbr.to_csv(path_csv, index=False, encoding='utf-8-sig')

    print('Case base disimpan:')
    print('  JSON : ' + path_json)
    print('  CSV  : ' + path_csv)
    print('  Total: ' + str(len(case_base)) + ' kasus')
    print()
    display(df_cbr.head())

Case base disimpan:
  JSON : ../data/processed/case_base\case_base_20260627_222843.json
  CSV  : ../data/processed/case_base\case_base_20260627_222843.csv
  Total: 55 kasus



,case_id,nomor,tanggal,pengadilan,para_pihak,kategori,problem,solution,dasar_hukum,kata_kunci,url
0,putusan_1005_pdt.p_2025_pn_tng_20260627213325,,,,,,pemohon,perkara – perkara\nd\ng\nperdata Permohonan pa...,,"pemohon, mahkamah, nama, surat, ridwan, putusa...",
1,putusan_114_pdt.p_2026_pn_mpw_20260627210712,,,,,,PERKARA\nR\na Menimbang bahwa Pemohon dengan s...,pegrkara perdata pada peradilan tingkat pertam...,,"pemohon, mahkamah, tanggal, anak, agung, infor...",
2,putusan_120_pdt.p_2026_pn_rbi_20260627210225,,,,,,"PERKARA\nh k\nMenimbang, bahwa Para Pemohon de...","o perkara\nperdata pada tingkat pertama, telah...",Pasal 193n\nn\nRBg,"pemohon, mahkamah, tanggal, lahir, anak, suci,...",
3,putusan_122_pdt.p_2026_pn_rbi_20260627210319,,,,,,"PERKARA\nh k\nMenimbang, bahwa Para Pemohon de...","o perkara\nperdata pada tingkat pertama, telah...",Pasal 193n\nn\nRBg,"pemohon, mahkamah, tanggal, lahir, anak, suci,...",
4,putusan_128_pdt.p_2026_pn_rbi_20260627210328,,,,,,PERKARA i\nl\nMenimbang bahwa Para Pemohon den...,"Perkara Perdata pada\nd\ng\ntingkat pertama, t...",Pasal 1865 Kitab Undang-Undang Hukum Perdata\n...,"pemohon, mahkamah, tahun, anak, informasi, agu...",
